In [2]:
from pathlib import Path
import pandas as pd
import requests
import numpy as np

In [4]:
#Estimating gross revenue for london ymd  = '20260619'

#Import Listings csv, ensure correct comprehension of price variable by removing symbols/commas and converting to a float
#low memory false to avoid mixed-type warnings


YMD = '20260917'
CITY = 'London'
HOST = 'https://orca.casa.ucl.ac.uk'
BASE_URL = f'{HOST}/~jreades/data'
CACHE_DIR = Path('data')
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def fetch_file(filename: str, cache_dir: Path = CACHE_DIR, timeout: int = 60) -> Path:
    """Download a file to the cache directory if not already present; return its path."""
    dest = cache_dir / filename
    if dest.exists():
        print(f"Using cached {filename}")
        return dest

    url = f'{BASE_URL}/{filename}'
    print(f"Downloading {filename}")
    tmp = dest.with_suffix(dest.suffix + '.part')
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    tmp.rename(dest)  # atomic: no half-written file is ever mistaken for a good cache
    return dest


listings_path = fetch_file(f'{YMD}-{CITY}-listings.csv.gz')
calendar_path = fetch_file(f'{YMD}-{CITY}-calendar.csv.gz')

Listings = pd.read_csv(listings_path, low_memory=False)

# load only columns to use for now
calendar = pd.read_csv(calendar_path, compression="gzip",
                  usecols=["listing_id", "date", "available"])

# ensure price is a float and remove currency symbols and commas
# bias introduced: price at time of scrape is applied for the whole year though prices vary seasonally ?? SOURCE ON EXTENT OF VARIATION IN LONDON
Listings["price"] = Listings["price"].replace(r"[\$£,]", "", regex=True).astype(float)

# Keep active listings only by selecting for listings with reviews greater than 1 and listings that have a price
# Bias introduced: listings without reviews are dropped, additionally given data explored later in this file, ...
#... having no price could reflect a flaw in scraping not the true listing. The effect of both is to lower the overall total.
Listings = Listings[(Listings["number_of_reviews_ltm"] > 0) & Listings["price"].notna()].copy()

Using cached 20260917-London-listings.csv.gz
Using cached 20260917-London-calendar.csv.gz


In [5]:
# clarifying time period with dates from within data as opposed to file name dates for accuracy
# in case the scrape spans a few days, use latest date
scraped = pd.to_datetime(Listings["last_scraped"])
# remove # to check scraped window
#print(scraped.min(), scraped.max())
#print(scraped.value_counts().head())
end = scraped.max()
start = end - pd.DateOffset(years=1)
# creating object for future use in print functions
period_label = f"{start:%d %B %Y} to {end:%d %B %Y}"
print(f"Estimate period: {period_label}")

Estimate period: 23 September 2025 to 23 September 2026


In [26]:
# Investigating price outliers
# although most dead listings have already been removed through the reviews filter
# some hosts use high prices to keep a listing active but ensure they don't get any bookings (FIND/INSERT SOURCE)
# room-type is an important variable as different price ranges associated with entire home vs. private room
def price_exploration(Listings, res):
    print(Listings["price"].describe(percentiles=[.5, .9, .95, .99, .999]))
    cols = ["room_type", "neighbourhood_cleansed", "price",
            "minimum_nights", "number_of_reviews_ltm", "calculated_host_listings_count"]
    print(Listings.nlargest(10, "price")[cols])
    print(Listings.nsmallest(10, "price")[cols])
    top20_share = res.nlargest(20, "revenue")["revenue"].sum() / res["revenue"].sum()
    print(f"Top 20 listings account for {top20_share:.1%} of estimated revenue")

# not a huge impact on result, but remove to make result more defensible, also may have more impact if want to do spatial analysis at level of e.g. ward
# low price rooms seem plausible, and may have many reviews, so only the high end is flagged
# several high price rooms belong to hosts with many listings (e.g. sample data include a host with 84 listings), but the diagnostics below show this is not a reliable indicator on its own
# therefore extreme prices are removed only where there is also little evidence of real bookings (reviews)


# create function that flags high prices
# use logs as price is right-skewed
# use median/MAD instead of mean/sd to mitigate outliers impact
# initially use z-score cut off of 4 (k), 200 is minimum number of listings required of a room type to supply stats
def flag_high_price(Listings, k=4.0, min_n=200):
    lp = np.log(Listings["price"])
    g = Listings["room_type"]

    med = lp.groupby(g).transform("median")     # add room type med log price benchmark to each listing
    mad = (lp - med).abs().groupby(g).transform("median") * 1.4826    # mad scaled so comparable to sd

    # small groups (hotel, shared) borrow pooled stats
    small = g.map(g.value_counts()) < min_n
    p_med = lp.median()
    p_mad = (lp - p_med).abs().median() * 1.4826
    med = med.where(~small, p_med)
    mad = mad.where(~small, p_mad)

    z = (lp - med) / mad
    return z > k          # high end only


# tighter price cut off on entire homes due to wider price range observed
# no. reviews seemed to not have massive effect on outcome, 2 reviews chosen over 3 reviews as threshold
# use the combination of extreme price and little evidence of real bookings (reviews) to flag high prices
def remove_price_outliers(Listings, k_home=5.0, k_other=4.0, max_reviews=2, min_n=200):
    is_home = Listings["room_type"].eq("Entire home/apt")
    flagged = pd.Series(
        np.where(is_home,
                 flag_high_price(Listings, k_home, min_n),
                 flag_high_price(Listings, k_other, min_n)),
        index=Listings.index)
    drop = flagged & (Listings["number_of_reviews_ltm"] <= max_reviews)
    return Listings[~drop].copy(), Listings[drop].copy()      # (cleaned, dropped)


# --- exploratory diagnostics used to choose k (run on the pre-removal data) ---
def price_outlier_diagnostics(Listings):
    l = Listings.copy()
    l["flag4"] = flag_high_price(l, k=4)
    l["flag5"] = flag_high_price(l, k=5)

    # what price does the cutoff imply per room type
    print(l.groupby("room_type")["price"]
            .agg(median="median", p99=lambda s: s.quantile(.99), max="max"))

    # print smallest price flagged by function to check if reasonable
    print(l.loc[l["flag4"]].groupby("room_type")["price"].agg(["count", "min"]))

    zone = l[l["flag4"] & ~l["flag5"]]
    print(zone[["room_type", "price", "number_of_reviews_ltm",
                "calculated_host_listings_count", "neighbourhood_cleansed"]]
          .sort_values("price"))

    # are flagged listings disproportionately multi-listing hosts?
    print(l.groupby("flag4")["calculated_host_listings_count"]
            .agg(["median", lambda s: (s >= 10).mean()]))

    # output suggests many are high prices are legit as they've many reviews, multilisting hosts also doesn't seem a useful indicator (previously stated professional operators may use price to manage bookings)
    # entire home/apt has much wider range of prices (due to nature) than private room/shared room etc., so seems require different sensitivities


# --- exploratory testing no. review threshold on data ---
def review_threshold_sensitivity(Listings, params, thresholds=(1, 2, 3)):
    rows = []
    for t in [None, *thresholds]:
        l = Listings if t is None else remove_price_outliers(Listings, max_reviews=t)[0]
        r = estimate(l, **params)
        rows.append({"review_threshold": t or "none", "listings": len(r),
                     "revenue_£m": round(r["revenue"].sum() / 1e6, 2)})
    out = pd.DataFrame(rows)
    out["change_vs_none_%"] = (out["revenue_£m"] / out["revenue_£m"].iloc[0] - 1) * 100
    return out


# --- run ---
res = estimate(Listings, **params)          
price_exploration(Listings, res)
price_outlier_diagnostics(Listings)
print(review_threshold_sensitivity(Listings, params))

Listings_clean, dropped = remove_price_outliers(Listings, k_home=5, k_other=4, max_reviews=2)
print(f"Dropped {len(dropped)} listings ({len(dropped)/len(Listings):.2%})")
print(dropped.groupby("room_type")["price"].agg(["count", "min"]))
print(dropped["neighbourhood_cleansed"].value_counts())


# Listings with extreme prices relative to their room type (robust z > 5 on log price for entire homes,
# > 4 for other types; effective cutoffs of about £3,350 and £440) were excluded where they also had two or
# fewer reviews in the last 12 months. This removed 24 listings (0.06%). Alternative review thresholds of 1
# and 3 remove 18 and 29 listings respectively, and the effect on estimated revenue is -0.19% (-0.17% and -0.27% for thresholds of 1 and 3).

count    40129.000000
mean       218.992723
std        366.707161
min         15.000000
50%        173.260000
90%        402.500000
95%        521.868000
99%        918.922400
99.9%     1961.752480
max      59214.000000
Name: price, dtype: float64
             room_type  neighbourhood_cleansed     price  minimum_nights  \
59223     Private room                 Lambeth  59214.00             1.0   
61237  Entire home/apt                Haringey  10070.50             2.0   
79766  Entire home/apt  Hammersmith and Fulham  10015.71             1.0   
24891  Entire home/apt           Tower Hamlets  10000.00             1.0   
54824     Private room               Southwark   8500.00             2.0   
24902  Entire home/apt           Tower Hamlets   8000.00             1.0   
82587  Entire home/apt             Westminster   7040.00             7.0   
50442  Entire home/apt                   Brent   5758.00             1.0   
78459  Entire home/apt             Westminster   5020.00            

In [28]:


# Assumptions are included as a parameter object so they can be adjusted to create a range of possible totals that acknowledge uncertainty
# These include: the review rate as indicator of total occupation rate (Airbnb say 50%, but other studies suggest 30-70%) LONDON SPECIFIC RATE? , average stay length for London is XX,
# total cap on 90 days for London, a plausible cap for a non-london context (given evidence on abuse of cap) ??

params = dict(review_rate=0.5, avg_stay=3, home_cap=90, other_cap=255)

# 90 night cap is an annual cap so fits annual nature of the data
# On 22nd June 2026, fee process changed from a split of guest 3% and host 12-14.2% to a united 15% fee on hosts, if any future analyses use this process, adjust code

def estimate(Listings_clean, review_rate=0.5, avg_stay=3, home_cap=90, other_cap=255):
    l = Listings_clean.copy()
    
    # using long term reviews as a better annual signal than reviews_per_month
    # estimate no. bookings
    
    l["bookings"] = l["number_of_reviews_ltm"] / review_rate
    # estimate total number of nights
    # using average stay length as a default, but if listing has a minimum nights per stay, using which ever is larger
    # bias introduced: listings with long minimums may be understated
    l["stay_len"] = l["minimum_nights"].clip(lower=avg_stay)
    l["nights"] = l["bookings"] * l["stay_len"]
    # apply an occupancy cap based on room type (entire listings have the london 90 day cap)
    # bias: assumes compliance with the 90 day cap
    cap = l["room_type"].eq("Entire home/apt").map({True: home_cap, False: other_cap})
    l["nights"] = l["nights"].clip(upper=cap)
    l["revenue"] = l["nights"] * l["price"]
    return l

res = estimate(Listings_clean, **params)     
print(f"Relevant listings: {len(res):,}")
print(
    f"Est. gross host revenue for {period_label}: £{res['revenue'].sum():,.0f}\n"
    f"Parameters: review rate = {params['review_rate']:.0%}, "
    f"avg stay = {params['avg_stay']} nights (floored at each listing's minimum_nights), "
    f"annual night caps: entire homes = {params['home_cap']}, other room types = {params['other_cap']} | "
    f"source: {YMD}-{CITY}-listings.csv.gz"
)
print(f"Est. Airbnb take (~14%):  £{res['revenue'].sum() * 0.14:,.0f}")



Active listings: 40,105
Est. gross host revenue for 23 September 2025 to 23 September 2026: £450,127,344
Parameters: review rate = 50%, avg stay = 3 nights (floored at each listing's minimum_nights), annual night caps: entire homes = 90, other room types = 255 | source: 20260917-London-listings.csv.gz
Est. Airbnb take (~14%):  £63,017,828


In [29]:
# how much do the model assumptions matter
for rr in (0.3, 0.5, 0.72):
    for stay in (2, 3, 4):
        p = {**params, "review_rate": rr, "avg_stay": stay}
        total = estimate(Listings, **p)["revenue"].sum()
        print(f"review rate={rr:.0%}, avg stay={stay}: £{total/1e6:,.0f}m")

review rate=30%, avg stay=2: £505m
review rate=30%, avg stay=3: £577m
review rate=30%, avg stay=4: £635m
review rate=50%, avg stay=2: £377m
review rate=50%, avg stay=3: £451m
review rate=50%, avg stay=4: £514m
review rate=72%, avg stay=2: £291m
review rate=72%, avg stay=3: £361m
review rate=72%, avg stay=4: £423m


In [13]:
#Cross check occupancy against calendar : review-rate method should be below calendar occupancy as
#calendar not hugely accurate as hosts blocl dates as unavailable for maintenance, personal use etc.
#calendar cover the year after the scrape, not data reviewed - and hosts may change calendar throughout the year

#counting unavailable days per listing
# True where day is unavailable added as new column (called blocked)
# group by to have one group per listing
# create 2 values per listing - days (calendar rows 365) and unavailable (sum of blocked column, as True = 1)
# results in occ  which has one row per listing, indexed by listing_id
# calender_occ is share of days unavailble, cal_nights is rescaled to a 365 night year (rescaling as some listings don't have 365 calendar rows)
# dividing by days to enable comparison
occ = (calendar.assign(blocked=calendar["available"].eq("f"))
          .groupby("listing_id")
          .agg(days=("date", "count"), unavailable=("blocked", "sum")))
occ["calendar_nights"] = occ["unavailable"] / occ["days"] * 365   # annualised
occ["calendar_occ"] = occ["unavailable"] / occ["days"]

# left join using shared id column
# then convert reviews per night into an occupancy share to compare with calender_occ

chk = res.merge(occ, left_on="id", right_index=True, how="left")
chk["review_occ"] = chk["nights"] / 365

print(chk[["review_occ", "calendar_occ"]].describe())
print("Listings where review-based nights exceed calendar nights:",
      f"{(chk['nights'] > chk['calendar_nights']).mean():.1%}")

chk["capped"] = chk["nights"].ge(chk["room_type"].eq("Entire home/apt").map({True: 90, False: 255}) - 1e-9)

summary = (chk.groupby("room_type")
              .agg(n=("id", "count"),
                   review_occ=("review_occ", "median"),
                   calendar_occ=("calendar_occ", "median"),
                   share_capped=("capped", "mean"),
                   share_above_ceiling=("nights", lambda s: (s > chk.loc[s.index, "calendar_nights"]).mean())))
print(summary)

#results show that at the market level, review based model sits well below the calendar on averages
# however a significant proportion exceed the calendar cieling (review model shows more booked nights than the calendar shows is available)

#Review-based occupancy is lower than calendar blocking at the market level, as expected, since blocked days include non-bookings. However, 28% 
#of listings show review-implied nights above calendar-unavailable nights. The 75th percentile 
#coincides with the 90-night cap, so it reflects censoring, not necessarily host behaviour.
# possible causes: time windows differ, far future dates often not yet booked, potentially assumptions around review rate thresholds is inflating some listings
# also calendar spread is huge so likely calendar_occupancy is dominated by hosts blocking days out therefore not superior to review-based occupancy assumption




         review_occ  calendar_occ
count  39831.000000  40129.000000
mean       0.164208      0.422814
std        0.150987      0.322417
min        0.016438      0.000000
25%        0.049315      0.117808
50%        0.131507      0.356164
75%        0.246575      0.731507
max        0.698630      0.997260
Listings where review-based nights exceed calendar nights: 28.2%
                     n  review_occ  calendar_occ  share_capped  \
room_type                                                        
Entire home/apt  27864    0.115068      0.361644      0.283520   
Hotel room          42    0.032877      0.036986      0.071429   
Private room     12110    0.147945      0.342466      0.092568   
Shared room        113    0.147945      0.172603      0.115044   

                 share_above_ceiling  
room_type                             
Entire home/apt             0.251220  
Hotel room                  0.500000  
Private room                0.349711  
Shared room                 0.442478 

count    40129.000000
mean       218.992723
std        366.707161
min         15.000000
50%        173.260000
90%        402.500000
95%        521.868000
99%        918.922400
99.9%     1961.752480
max      59214.000000
Name: price, dtype: float64
             room_type  neighbourhood_cleansed     price  minimum_nights  \
59223     Private room                 Lambeth  59214.00             1.0   
61237  Entire home/apt                Haringey  10070.50             2.0   
79766  Entire home/apt  Hammersmith and Fulham  10015.71             1.0   
24891  Entire home/apt           Tower Hamlets  10000.00             1.0   
54824     Private room               Southwark   8500.00             2.0   
24902  Entire home/apt           Tower Hamlets   8000.00             1.0   
82587  Entire home/apt             Westminster   7040.00             7.0   
50442  Entire home/apt                   Brent   5758.00             1.0   
78459  Entire home/apt             Westminster   5020.00            